# 🏃 Tango Benchmark — **Notebook C** (Seed 2)

### What this notebook runs:
| Optimizer | LR | Seed | ~Time |
|-----------|-----|------|-------|
| Adam | 1e-3 | 2 | ~20 min |
| Adam | 3e-4 | 2 | ~20 min |
| Adam | 1e-4 | 2 | ~20 min |
| Lion | 1e-4 *(default best LR)* | 2 | ~20 min |
| Tango | — | 2 | ~22 min |

**Total ≈ 1.7 hours on a T4 GPU.**

### Instructions:
1. `Runtime → Change runtime type → T4 GPU`
2. Run all cells top-to-bottom.
3. At the end, **copy the printed JSON block** into a text file called `results_C.txt`.
4. Share `results_C.txt` with the person running the **Combiner notebook**.

In [ ]:
!pip install -q torch torchvision
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only ⚠️')
print('CUDA:', torch.version.cuda)

In [ ]:
import copy, json, math, os, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as transforms

THIS_SEED  = 2
N_EPOCHS   = 50
BATCH_SIZE = 128
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ADAM_LRS   = [1e-3, 3e-4, 1e-4]
LION_LR    = 1e-4

LR_MAX=0.005; LR_MIN=0.000001; T_CYCLE=800; EXPLORE_FRAC=0.8
LR_EXPLOIT_START=0.005; LR_EXPLOIT_END=0.0005
EXPLORE_DECAY_START=0.6; EXPLORE_FLOOR=0.03
TANG_BETA=0.80; TANG_EPS_BASE=4e-4; TANG_INTERVAL=100; TANG_LOSS_GATE=0
SHARP_UPDATE_INT=200; FD_EPS=1e-4; NOISE_SCALE=2e-4; NOISE_START=500

print(f'Notebook C | seed={THIS_SEED} | device={DEVICE}')

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
train_set = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform_train)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
def make_loaders(seed):
    g = torch.Generator(); g.manual_seed(seed)
    return (torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, generator=g),
            torch.utils.data.DataLoader(test_set,  batch_size=256,        shuffle=False, num_workers=2))
print('Data ready ✓')

In [ ]:
def make_resnet18():
    m = torchvision.models.resnet18(weights=None)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(512, 10)
    return m.to(DEVICE)
print(f'ResNet-18 params: {sum(p.numel() for p in make_resnet18().parameters())/1e6:.2f}M')

In [ ]:
class Lion(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9,0.99), weight_decay=0.0):
        super().__init__(params, dict(lr=lr, betas=betas, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad=p.grad; state=self.state[p]
                if len(state)==0: state['exp_avg']=torch.zeros_like(p)
                exp_avg=state['exp_avg']; b1,b2=group['betas']
                if group['weight_decay']!=0: p.data.mul_(1-group['lr']*group['weight_decay'])
                update=exp_avg*b1+grad*(1-b1); p.add_(torch.sign(update),alpha=-group['lr'])
                exp_avg.mul_(b2).add_(grad,alpha=1-b2)
        return loss

def cyclic_lr(step,T,lr_max,lr_min): return lr_min+0.5*(lr_max-lr_min)*(1+np.cos(np.pi*(step%T)/T))
def cosine_decay_lr(step,total,s,e):
    p=min(step/max(total,1),1.0); return e+0.5*(s-e)*(1+np.cos(np.pi*p))
def get_ef(step,total,ds_frac,floor):
    ds=int(total*ds_frac)
    if step<ds: return 1.0
    return max(floor,1.0-(step-ds)/max(total-ds,1))
def get_flat_grad(model):
    return np.concatenate([p.grad.detach().cpu().float().view(-1).numpy() if p.grad is not None
                           else np.zeros(p.numel(),dtype=np.float32) for p in model.parameters()]).astype(np.float32)
def get_flat_params(model):
    return np.concatenate([p.detach().cpu().float().view(-1).numpy() for p in model.parameters()]).astype(np.float32)
def set_flat_params(model,flat):
    o=0
    for p in model.parameters():
        n=p.numel(); p.data.copy_(torch.tensor(flat[o:o+n],dtype=p.dtype,device=p.device).view(p.shape)); o+=n
def fd_curvature(model,xb,yb,g_np,eps=FD_EPS):
    norm_g=np.linalg.norm(g_np)
    if norm_g<1e-12: return 0.0
    g_hat=(g_np/norm_g).astype(np.float32); p0=get_flat_params(model); crit=nn.CrossEntropyLoss()
    with torch.no_grad(): f0=crit(model(xb),yb).item()
    set_flat_params(model,p0+g_hat*eps)
    with torch.no_grad(): fe=crit(model(xb),yb).item()
    set_flat_params(model,p0)
    return float((fe-f0-eps*norm_g)/(0.5*eps**2+1e-30))
class TangentMomentum:
    def __init__(self,dim,beta=TANG_BETA): self.beta=beta; self.v=np.zeros(dim,dtype=np.float32)
    def step(self,model,g_np,epsilon):
        g_hat=g_np/(np.linalg.norm(g_np)+1e-8); noise=np.random.randn(len(g_np)).astype(np.float32)
        noise-=np.dot(noise,g_hat)*g_hat; noise/=np.linalg.norm(noise)+1e-8
        self.v=self.beta*self.v+(1-self.beta)*noise; d=self.v/(np.linalg.norm(self.v)+1e-8)
        p0=get_flat_params(model); set_flat_params(model,p0+d*epsilon)
print('Helpers defined ✓')

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct=total=0; total_loss=0.0; crit=nn.CrossEntropyLoss()
    for xb,yb in loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE); out=model(xb); loss=crit(out,yb)
        total_loss+=loss.item()*yb.size(0); correct+=(out.argmax(1)==yb).sum().item(); total+=yb.size(0)
    model.train(); return total_loss/total, correct/total

def _train_loop(seed, optimizer_fn, label):
    torch.manual_seed(seed); np.random.seed(seed)
    train_loader,test_loader=make_loaders(seed)
    model=make_resnet18(); opt=optimizer_fn(model); crit=nn.CrossEntropyLoss()
    best_loss=float('inf'); best_acc=0.0; history=[]; t0=time.time()
    print(f'  → {label} seed={seed}')
    for epoch in range(1,N_EPOCHS+1):
        model.train(); ep_loss=0.0; ep_n=0
        for xb,yb in train_loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad()
            loss=crit(model(xb),yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            ep_loss+=loss.item()*yb.size(0); ep_n+=yb.size(0)
        vl,va=evaluate(model,test_loader)
        if vl<best_loss: best_loss=vl; best_acc=va
        history.append({'epoch':epoch,'train_loss':ep_loss/ep_n,'val_loss':vl,'val_acc':va})
        if epoch%10==0: print(f'    epoch {epoch:3d}  val_loss={vl:.4f}  val_acc={va:.4f}')
    elapsed=time.time()-t0
    print(f'  ✓ {elapsed/60:.1f} min | best={best_loss:.4f}  acc={best_acc:.4f}')
    return {'seed':seed,'final_val_loss':vl,'final_val_acc':va,'best_val_loss':best_loss,'best_val_acc':best_acc,'elapsed_sec':elapsed,'history':history}

def run_adam(seed,lr):
    r=_train_loop(seed,lambda m:torch.optim.Adam(m.parameters(),lr=lr,betas=(0.9,0.999)),f'Adam lr={lr:.0e}')
    r['optimizer']='adam'; r['lr']=lr; return r
def run_lion(seed,lr):
    r=_train_loop(seed,lambda m:Lion(m.parameters(),lr=lr,betas=(0.9,0.99),weight_decay=1e-4),f'Lion lr={lr:.0e}')
    r['optimizer']='lion'; r['lr']=lr; return r
print('run_adam/lion defined ✓')

In [ ]:
def run_tango(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    train_loader,test_loader=make_loaders(seed)
    model=make_resnet18()
    opt=torch.optim.AdamW(model.parameters(),lr=LR_MAX,betas=(0.9,0.999),weight_decay=0.0)
    crit=nn.CrossEntropyLoss()
    n_params=sum(p.numel() for p in model.parameters())
    tang_mom=TangentMomentum(dim=n_params)
    spe=len(train_loader); total_steps=N_EPOCHS*spe; explore_end=int(total_steps*EXPLORE_FRAC)
    best_val_loss=float('inf'); best_val_acc=0.0; best_state=copy.deepcopy(model.state_dict())
    sharpness=0.0; tang_exec=0; tang_block=0; history=[]; step=0; t0=time.time()
    print(f'  → Tango seed={seed} | {n_params/1e6:.2f}M | {total_steps} steps')
    for epoch in range(1,N_EPOCHS+1):
        model.train(); ep_loss=0.0; ep_n=0
        for xb,yb in train_loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            lr=(cyclic_lr(step,T_CYCLE,LR_MAX,LR_MIN) if step<explore_end
                else cosine_decay_lr(step-explore_end,total_steps-explore_end,LR_EXPLOIT_START,LR_EXPLOIT_END))
            for pg in opt.param_groups: pg['lr']=lr
            ef=get_ef(step,total_steps,EXPLORE_DECAY_START,EXPLORE_FLOOR)
            opt.zero_grad(); loss=crit(model(xb),yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0)
            g_np=get_flat_grad(model); opt.step(); loss_val=loss.item()
            ep_loss+=loss_val*yb.size(0); ep_n+=yb.size(0)
            if step>=NOISE_START:
                sigma=NOISE_SCALE*np.sqrt(max(loss_val,1e-8))*ef
                with torch.no_grad():
                    for p in model.parameters(): p.data.add_(torch.randn_like(p)*sigma)
            if step%SHARP_UPDATE_INT==0 and step>0: sharpness=fd_curvature(model,xb,yb,g_np)
            if step%TANG_INTERVAL==0 and step>500:
                if loss_val>TANG_LOSS_GATE*max(best_val_loss,1e-10) and sharpness<0.8*(2.0/max(lr,1e-8)):
                    ss=max(abs(sharpness),1.0); et=float(np.clip(TANG_EPS_BASE*(50.0/ss)**0.5,1e-5,5e-3))*ef
                    tang_mom.step(model,g_np,et); tang_exec+=1
                else: tang_block+=1
            step+=1
        vl,va=evaluate(model,test_loader)
        if vl<best_val_loss: best_val_loss=vl; best_val_acc=va; best_state=copy.deepcopy(model.state_dict())
        history.append({'epoch':epoch,'train_loss':ep_loss/ep_n,'val_loss':vl,'val_acc':va,'lr':lr,'tang_exec':tang_exec,'tang_block':tang_block})
        if epoch%10==0:
            tr=tang_exec/max(tang_exec+tang_block,1)
            print(f'    epoch {epoch:3d}  val_loss={vl:.4f}  val_acc={va:.4f}  tang_rate={tr:.2%}')
    model.load_state_dict(best_state); fvl,fva=evaluate(model,test_loader)
    elapsed=time.time()-t0; tr=tang_exec/max(tang_exec+tang_block,1)
    print(f'  ✓ {elapsed/60:.1f} min | best={best_val_loss:.4f}  acc={best_val_acc:.4f}  tang={tr:.2%}')
    return {'optimizer':'tango','seed':seed,'final_val_loss':fvl,'final_val_acc':fva,
            'best_val_loss':best_val_loss,'best_val_acc':best_val_acc,
            'tang_exec':tang_exec,'tang_block':tang_block,'tang_rate':tr,'elapsed_sec':elapsed,'history':history}
print('run_tango() defined ✓')

In [ ]:
grand_start=time.time()
results={'notebook':'C','seed':THIS_SEED,'adam':[],'lion':[],'tango':[]}
print('='*60); print(f'  NOTEBOOK C — SEED {THIS_SEED}'); print('='*60)
print('\n── Adam (3 LRs) ──')
for lr in ADAM_LRS: results['adam'].append(run_adam(THIS_SEED,lr))
print(f'\n── Lion (lr={LION_LR:.0e}) ──')
results['lion'].append(run_lion(THIS_SEED,LION_LR))
print('\n── Tango ──')
results['tango'].append(run_tango(THIS_SEED))
results['total_elapsed_sec']=time.time()-grand_start
print(f'\n✅ Notebook C DONE in {results["total_elapsed_sec"]/3600:.2f} hours')

In [ ]:
def slim(r):
    s={k:v for k,v in r.items() if k!='history'}
    s['history_summary']=[{'epoch':h['epoch'],'val_loss':h['val_loss'],'val_acc':h['val_acc']} for h in r.get('history',[])]
    return s
slim_results={'notebook':results['notebook'],'seed':results['seed'],
              'total_elapsed_sec':results['total_elapsed_sec'],
              'adam':[slim(r) for r in results['adam']],
              'lion':[slim(r) for r in results['lion']],
              'tango':[slim(r) for r in results['tango']]}
output_str=json.dumps(slim_results,indent=2)
with open('results_C.txt','w') as f: f.write(output_str)
print('\n'+'='*60)
print('  📋 COPY EVERYTHING BELOW THIS LINE INTO results_C.txt')
print('='*60); print(output_str); print('='*60)
print('File also saved as results_C.txt — download from Colab Files panel.')